# Projet RAG pas a pas - Assistant culinaire

Ce notebook explique comment construire un assistant RAG a partir des PDFs du dossier `Data/`. L'objectif est de comprendre chaque brique : ingestion, chunking, embeddings, recherche vectorielle, generation et evaluation.

## 1. Preparation

On charge les variables d'environnement et on verifie que les PDFs sont presents.

In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
DATA_DIR = PROJECT_ROOT / "Data"
PDFS = sorted(DATA_DIR.glob("*.pdf"))
PDFS

## 2. Extraction des pages PDF

Chaque page devient un document avec du texte et des metadonnees : fichier, volume et numero de page.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"

from src.ingestion import extract_pdf_pages

pages = extract_pdf_pages(DATA_DIR)
len(pages), pages[0]

In [ ]:
print(pages[0].text[:1000])

## 3. Decoupage en chunks

Le RAG ne donne pas tout le PDF au modele. Il decoupe le corpus en passages plus petits, avec un leger chevauchement pour ne pas casser le contexte.

In [ ]:
from src.chunking import build_chunks, count_tokens

chunks = build_chunks(pages, chunk_size=800, overlap=150)
len(chunks), chunks[0]

In [ ]:
print("Tokens du premier chunk:", count_tokens(chunks[0].text))
print(chunks[0].text[:1000])

## 4. Creation des embeddings

Les embeddings transforment chaque chunk en vecteur numerique. Cette cellule appelle l'API OpenAI et necessite `OPENAI_API_KEY` dans `.env`.

In [ ]:
from src.vector_store import embed_texts

# Pour tester vite, commence avec un petit echantillon.
sample_chunks = chunks[:10]
embeddings = embed_texts([chunk.text for chunk in sample_chunks])
embeddings.shape

## 5. Indexation FAISS

FAISS permet de retrouver rapidement les chunks les plus proches d'une question.

In [ ]:
from src.vector_store import save_index, load_index, search

# Index complet : decommenter quand tu es prete a indexer tout le corpus.
# embeddings = embed_texts([chunk.text for chunk in chunks])
# save_index(chunks, embeddings, "storage/faiss_index")

# Index de demonstration sur 10 chunks.
save_index(sample_chunks, embeddings, "storage/demo_index")
index, stored_chunks = load_index("storage/demo_index")
len(stored_chunks)

## 6. Recherche vectorielle

On pose une question, puis on inspecte les passages recuperes avant de generer une reponse.

In [ ]:
question = "Donne-moi une recette avec du poulet."
results = search(question, index, stored_chunks, top_k=3)
for result in results:
    print(result.score, result.chunk.volume, result.chunk.page, result.chunk.source_file)
    print(result.chunk.text[:500])
    print("---")

## 7. Pipeline RAG complet

Pour utiliser tout le corpus, lance d'abord `python build_index.py` dans un terminal. Ensuite, cette cellule charge l'index complet et genere une reponse avec citations.

In [ ]:
from src.rag_pipeline import answer_question

# Necessite un index complet cree par: python build_index.py
answer = answer_question("Trouve un dessert avec des fraises.", top_k=5)
print(answer.answer)
[(s.chunk.volume, s.chunk.page, s.score) for s in answer.sources]

## 8. Evaluation qualitative

Questions utiles pour tester le projet :

- Donne-moi une recette avec du poulet.
- Trouve un dessert avec des fraises.
- Quelle recette contient du brocoli ?
- Propose une recette de boisson ou punch.
- Quelle recette puis-je faire avec du fromage ?

Pour chaque question, verifie : pertinence des chunks, presence des sources, absence d'invention et clarte de la reponse.

In [ ]:
test_questions = [
    "Donne-moi une recette avec du poulet.",
    "Trouve un dessert avec des fraises.",
    "Quelle recette contient du brocoli ?",
    "Propose une recette de boisson ou punch.",
    "Quelle recette puis-je faire avec du fromage ?",
]

test_questions